<a href="https://colab.research.google.com/github/sumitp2703/FMML/blob/main/tranformer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
Implementation Of Transformers

Transformers are a type of deep learning model designed to handle sequential data, particularly effective in tasks involving language understanding and generation. They were introduced in the paper "Attention is All You Need" by Vaswani et al. in 2017 and have since become the foundation for many state-of-the-art models in NLP.
**Architecture of Transformers:**
Self-Attention Mechanism:
Transformer Blocks:
Transformers are composed of multiple identical layers called transformer blocks.
Each transformer block typically consists of:
*   **Multi-head Attention:** Simultaneously applies multiple self-attention mechanisms to capture different relationships between words.
Feedforward Neural Networks: After attention, each word's representation passes through a fully connected feedforward network.
*   **Layer Normalization**: Normalizes the output of each sub-layer before adding residual connections.





In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np


In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.encoding = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
        self.encoding[:, 0::2] = torch.sin(position * div_term)
        self.encoding[:, 1::2] = torch.cos(position * div_term)
        self.encoding = self.encoding.unsqueeze(0)

    def forward(self, x):
        return x + self.encoding[:, :x.size(1)].detach()


In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, hidden_dim, dropout):
        super(TransformerBlock, self).__init__()
        self.attention = nn.MultiheadAttention(d_model, num_heads, dropout=dropout)
        self.feedforward = nn.Sequential(
            nn.Linear(d_model, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attention_output, _ = self.attention(x, x, x, attn_mask=mask)
        x = x + self.dropout(self.norm1(attention_output))
        feedforward_output = self.feedforward(x)
        x = x + self.dropout(self.norm2(feedforward_output))
        return x


In [ ]:
class Transformer(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, hidden_dim, dropout, output_dim):
        super(Transformer, self).__init__()
        self.encoder = nn.Embedding(input_dim, d_model)
        self.pos_encoder = PositionalEncoding(d_model)
        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(d_model, num_heads, hidden_dim, dropout)
            for _ in range(num_layers)
        ])
        self.fc = nn.Linear(d_model, output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, src, src_mask=None):
        src = self.encoder(src) * np.sqrt(self.d_model)
        src = self.pos_encoder(src)
        for transformer in self.transformer_blocks:
            src = transformer(src, src_mask)
        output = self.fc(src[:, 0])
        return output


In [ ]:
def train(model, iterator, optimizer, criterion, clip):
    model.train()
    epoch_loss = 0
    for src, trg in iterator:
        optimizer.zero_grad()
        output = model(src)
        loss = criterion(output, trg)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
        optimizer.step()
        epoch_loss += loss.item()
    return epoch_loss / len(iterator)


In [ ]:
class Transformer(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, hidden_dim, dropout, output_dim, input_dim): # Add input_dim here
        super(Transformer, self).__init__()
        self.encoder = nn.Embedding(input_dim, d_model) # Use it here
        # ... rest of your code
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Initialize model
num_layers = 6
d_model = 512
num_heads = 8
hidden_dim = 2048
dropout = 0.1
output_dim = 10 # Example output dimension, adjust as needed
model = Transformer(num_layers, d_model, num_heads, hidden_dim, dropout, output_dim, 1000).to(device)

# Define optimizer and criterion
learning_rate = 0.001 # Example learning rate, adjust as needed
optimizer = optim.Adam(model.parameters(), lr=learning_rate)
criterion = nn.CrossEntropyLoss()

# Train the model
# Replace these with your actual training data
train_src = ["This is a source sentence.", "Another source sentence here."]
train_trg = ["This is a target sentence.", "Another target sentence here."]

for epoch in range(num_epochs):
    # Example - replace with your actual data loading logic
    train_data = list(zip(train_src, train_trg))
    train_iterator = iter(train_data)
    # ... rest of your code ...





In [ ]:
def evaluate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0
    with torch.no_grad():
        for src, trg in iterator:
            output = model(src) # Get model predictions
            loss = criterion(output, trg) # Calculate loss
            epoch_loss += loss.item()
    return epoch_loss / len(iterator)